In [ ]:
%pip install pandas -q
%pip install matplotlib -q
%pip install scipy -q

In [ ]:
from google.colab import drive
import pandas as pd
from matplotlib import pyplot as plt
import scipy
#drive.mount('/content/drive')

%matplotlib inline

In [ ]:
def clear_result(result):
  result = result.replace("[", "")
  result = result.replace("]", "")
  return result

def trim_result(row, result_name, average_name):
  column = row[f'{result_name}_avg_{average_name}'].split(" ")
  column = map(clear_result, column)
  column = list(filter(lambda x: x != '', column))
  if len(column) == 1:
    column = float(column[0])
  elif len(column) == 2:
    column = float(column[1])

  return column

def trim_results(row, average_name):
  precision = trim_result(row, 'precision', average_name)
  recall = trim_result(row, 'recall', average_name)
  f1 = trim_result(row, 'f1', average_name)
  row['average_type'] = average_name
  row['precision'] = precision
  row['recall'] = recall
  row['f1'] = f1

  hidden_labels = row['hidden_labels'].replace("[", '')
  hidden_labels = hidden_labels.replace("]", '')
  hidden_labels= hidden_labels.replace("'", "")
  hidden_labels = hidden_labels.split(",")

  row['hidden_labels'] = list(filter(lambda x : x != '', hidden_labels))
  row['hidden_labels_count'] = len(row['hidden_labels'])
  return row

In [ ]:
entity_translation = {
    'Ekosistem': 'Ecosystem',
    'Karasal Ekosistem': 'Terrestrial Ecosystem',
    'Yerleşim Yerleri': 'Settlements',
    'Sucul Ekosistem': 'Aquatic Ecosystem',
    'Kirletici': 'Pollutant',
    'Sıvı Kirletici': 'Liquid Pollutant',
    'Katı Kirletici': 'Solid Pollutant',
    'Gaz Kirletici': 'Gas Pollutant',
    'Enerji': 'Energy',
    'Afet': 'Phenomenon',
    'Doğal Afet': 'Natural Phenomenon',
    'İnsan Kaynaklı Afet': 'Human-Sourced Phenomenon',
    'Biota': 'Biota',
    'İnsan Dışı Biota': 'Non-Anthropogenic Biota',
    'Sucul Biota': 'Aquatic Biota',
    'Karasal Biota': 'Terrestrial Biota',
    'İnsan': 'Human',
    'Çevresel Etki': 'Environmental Impact',
    'Ekolojik Etki': 'Ecological Impact',
    'Refah Etkisi': 'Well-Being Impact',
    'Ekonomik Etki': 'Economical Impact',
    'Sağlık Etkisi': 'Health Impact',
    'Sosyal Etki': 'Social Impact',
    'Çevre Yönetimi': 'Environment Management',
    'Düzenleme': 'Regulation',
    'Azaltma': 'Mitigation',
    'Arıtım': 'Treatment',
    'Kirleten': 'Polluter',
    'İnsan Kaynaklı Kirleten': 'Human-Sourced Polluter',
    'Doğal Kirleten': 'Natural Polluter'
}

entity_translation_reverse = {v: k for k, v in entity_translation.items()}


def translate_hidden_labels(df):
  hidden_labels = [entity_translation[label.strip()] for label in df['hidden_labels'] ]
  return hidden_labels


In [ ]:
def t_test(df, column1, column2):
  df = df.dropna(subset=[column1, column2])
  t_stat, p_val = scipy.stats.mstats.ttest_rel(df[column1], df[column2], alternative='greater')
  return t_stat, p_val

def t_test_two_way(df, column1, column2):
  df = df.dropna(subset=[column1, column2])
  t_stat, p_val = scipy.stats.mstats.ttest_rel(df[column1], df[column2])
  return t_stat, p_val


In [ ]:
graph_prefix = "Distilbert - News Data"
#graph_prefix = "DistilbertNER - News Data"
#graph_prefix = "Distilbert - News and GPT Data"
#graph_prefix = "DistilbertNER - News and GPT Data"

#df = pd.read_csv('/content/drive/MyDrive/ner_results/distilbert-base-turkish-cased_17-07-2024-18-13_seed_117_version_101_news_data.csv')
#df = pd.read_csv('/content/drive/MyDrive/ner_results/distilbert-base-turkish-cased_17-07-2024-17-25_seed_117_version_101_news_and_gpt_data.csv')
#df = pd.read_csv('/content/drive/MyDrive/ner_results/turkish_ner_17-07-2024-19-30_seed_117_version_101_news_data.csv')
#df = pd.read_csv('/content/drive/MyDrive/ner_results/turkish_ner_17-07-2024-19-21_seed_117_version_101_news_and_gpt_data.csv')

#df = pd.read_csv('/content/drive/MyDrive/ner_results/averaged/distilbert_news_averaged.csv')
#df = pd.read_csv('/content/drive/MyDrive/ner_results/averaged/turkish_news_averaged.csv')
# df = pd.read_csv('/content/drive/MyDrive/ner_results/averaged/distilbert_news_and_gpt_averaged.csv')
# df = pd.read_csv('/content/drive/MyDrive/ner_results/averaged/turkish_news_and_gpt_averaged.csv')

df = pd.read_csv('/content/drive/MyDrive/ner_results/july_v2/distilbert-base-turkish-cased_version_101_news_only_without_validation.csv')
#df = pd.read_csv('/content/drive/MyDrive/ner_results/july_v2/turkish_ner_version_101_news_only_without_validation.csv')
#df = pd.read_csv('/content/drive/MyDrive/ner_results/july_v2/distilbert-base-turkish-cased_version_101_ner_and_gpt_data_without_validation.csv')
#df = pd.read_csv('/content/drive/MyDrive/ner_results/july_v2/turkish_ner_version_101_news_and_gpt_data_without_validation.csv')


result_names = [
    'precision_avg',
    'recall_avg',
    'f1_avg'
]

remove_result_averages = [
    'none',
    'binary',
    'macro',
    'weighted',
    'micro'
]

df_remove_results = []
for result_name in result_names:
  for remove_result_average in remove_result_averages:
    df_remove_results.append(f'{result_name}_{remove_result_average}')

df_remove_columns = [
    'model_checkpoint',
    'tokenizer_checkpoint',
    'hyponyms',
    'hypernyms',
    'siblings',
    'train_runtime',
    'train_steps_per_second',
    'train_loss'
]

df_remove_columns.extend(df_remove_results)

df['average_type'] = None
df['precision'] = None
df['recall'] = None
df['f1'] = None
df

df = df.apply(trim_results, axis=1, args=('none',))

df = df.drop(columns=df_remove_columns, axis=1)
df['class_unseen'] = df['class_unseen'].map(entity_translation)
df['hidden_labels'] = df.apply(translate_hidden_labels, axis=1)

df.head()

In [ ]:
df

In [ ]:
def apply_has_shot_flag(dataframe):
  has_one_shot = not dataframe[dataframe['shot_number'] == 1].empty
  dataframe['has_one_shot'] = has_one_shot

  has_ten_shot = not dataframe[dataframe['shot_number'] == 10].empty
  dataframe['has_ten_shot'] = has_ten_shot

  return dataframe

df = df.groupby("class_unseen").apply(apply_has_shot_flag)

In [ ]:
df.reset_index(drop=True, inplace=True)
df = df[df['has_one_shot']]

In [ ]:
criteria = (df['hidden_labels_count'] == 0) & (df['test_length'] > 1)
#shot_comp_df_with_relatives = df[criteria & (df['has_ten_shot'])]
shot_comp_df_with_relatives = df[criteria]

In [ ]:
#shot_comp_df_without_relatives = df[~criteria & (df['test_length'] > 1) & (df['has_ten_shot'])]
shot_comp_df_without_relatives = df[~criteria & (df['test_length'] > 1)]
shot_comp_df_without_relatives

In [ ]:
def shot_comparison_apply(dataframe):
  # class_unseen = dataframe['class_unseen'].iloc[0]
  zero_shot = dataframe[dataframe['shot_number'] == 0].iloc[0]

  one_shot_df = dataframe[dataframe['shot_number'] == 1]
  one_shot = None
  if not one_shot_df.empty:
    one_shot = dataframe[dataframe['shot_number'] == 1].iloc[0]

  ten_shot_df = dataframe[dataframe['shot_number'] == 10]
  ten_shot = None
  if not ten_shot_df.empty:
    ten_shot = dataframe[dataframe['shot_number'] == 10].iloc[0]

  series = pd.Series({
      'zero_shot_test_length': zero_shot['test_length'],
      'zero_shot_precision': zero_shot['precision'],
      'zero_shot_recall': zero_shot['recall'],
      'zero_shot_f1': zero_shot['f1'],
  })

  if one_shot is not None:
    series = pd.concat([series, pd.Series({
      'one_shot_precision': one_shot['precision'],
      'one_shot_precision_diff': one_shot['precision'] - zero_shot['precision'],
      'one_shot_recall': one_shot['recall'],
      'one_shot_recall_diff': one_shot['recall'] - zero_shot['recall'],
      'one_shot_f1': one_shot['f1'],
      'one_shot_f1_diff': one_shot['f1'] - zero_shot['f1']
    })])
  else:
    return None

  if ten_shot is not None:
    series = pd.concat([series, pd.Series({
      'ten_shot_precision': ten_shot['precision'],
      'ten_shot_precision_diff': ten_shot['precision'] - zero_shot['precision'],
      'ten_shot_recall': ten_shot['recall'],
      'ten_shot_recall_diff': ten_shot['recall'] - zero_shot['recall'],
      'ten_shot_f1': ten_shot['f1'],
      'ten_shot_f1_diff': ten_shot['f1'] - zero_shot['f1']
    })])
  else:
    series = pd.concat([series, pd.Series({
      'ten_shot_precision': float('nan'),
      'ten_shot_precision_diff': float('nan'),
      'ten_shot_recall': float('nan'),
      'ten_shot_recall_diff': float('nan'),
      'ten_shot_f1': float('nan'),
      'ten_shot_f1_diff': float('nan')
    })])

  return series


df_shot_comp_grouped_with_relatives = shot_comp_df_with_relatives.groupby('class_unseen').apply(shot_comparison_apply)

In [ ]:
df_shot_comp_relative_simple = df_shot_comp_grouped_with_relatives[['zero_shot_f1', 'one_shot_f1', 'ten_shot_f1']]
df_shot_comp_relative_simple

In [ ]:
df_shot_comp_relative_simple.describe()

In [ ]:
t_test(df_shot_comp_relative_simple, 'one_shot_f1', 'zero_shot_f1')

In [ ]:
t_test(df_shot_comp_relative_simple, 'ten_shot_f1', 'one_shot_f1')

In [ ]:
t_test(df_shot_comp_relative_simple, 'ten_shot_f1', 'zero_shot_f1')

In [ ]:
pd.DataFrame([
    ['One Shot - Zero Shot'] + list(t_test(df_shot_comp_relative_simple, 'one_shot_f1', 'zero_shot_f1')),
    ['Ten Shot - One Shot'] + list(t_test(df_shot_comp_relative_simple, 'ten_shot_f1', 'one_shot_f1')),
    ['Ten Shot - Zero Shot'] + list(t_test(df_shot_comp_relative_simple, 'ten_shot_f1', 'zero_shot_f1'))
    ], columns=['Test', 'T Test', 'P Value'])

In [ ]:
df_shot_comp_grouped_without_relatives = shot_comp_df_without_relatives.groupby('class_unseen').apply(shot_comparison_apply)
df_shot_comp_grouped_without_relatives

In [ ]:
df_shot_comp_without_relative_simple = df_shot_comp_grouped_without_relatives[['zero_shot_f1', 'one_shot_f1', 'ten_shot_f1']]
df_shot_comp_without_relative_simple

In [ ]:
df_shot_comp_without_relative_simple.describe()

In [ ]:
print('oneshot vs zeroshot', t_test(df_shot_comp_without_relative_simple, 'one_shot_f1', 'zero_shot_f1'))
print('tenshot vs oneshot', t_test(df_shot_comp_without_relative_simple, 'ten_shot_f1', 'one_shot_f1'))
print('tenshot vs zeroshot', t_test(df_shot_comp_without_relative_simple, 'ten_shot_f1', 'zero_shot_f1'))

In [ ]:
pd.DataFrame([
    ['One Shot - Zero Shot'] + list(t_test(df_shot_comp_without_relative_simple, 'one_shot_f1', 'zero_shot_f1')),
    ['Ten Shot - One Shot'] + list(t_test(df_shot_comp_without_relative_simple, 'ten_shot_f1', 'one_shot_f1')),
    ['Ten Shot - Zero Shot'] + list(t_test(df_shot_comp_without_relative_simple, 'ten_shot_f1', 'zero_shot_f1'))
    ], columns=['Test', 'T Test', 'P Value'])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def plot_grouped_bar_chart(df, graph_name):
    """
    Plots a grouped bar chart with the x-axis centered in the middle of the graph.

    Parameters:
    df (pd.DataFrame): DataFrame with 'one_shot_f1_diff' and 'ten_shot_f1_diff' columns.
                       The index of the DataFrame should be the class names.
    """
    # Check if the required columns are in the dataframe
    required_columns = ['one_shot_f1_diff', 'ten_shot_f1_diff']
    if not all(column in df.columns for column in required_columns):
        raise ValueError(f"DataFrame must contain the following columns: {required_columns}")

    # Replace NA values in ten_shot_f1_diff with a marker for missing data
    if 'ten_shot_f1_diff' in df:
      df['ten_shot_f1_diff'].fillna(value=np.nan, inplace=True)

    # Create an array of the indices for the x-axis, including padding
    indices = np.arange(len(df) + 1)  # Add one for the padding

    # Define the width of the bars
    bar_width = 0.35

    # Extract the class names from the index
    class_names = df.index.tolist()

    # Create the figure and axis
    fig, ax = plt.subplots(figsize=(10, 6))

    # Plot the bars for one_shot_f1_diff
    bars1 = ax.bar(indices[1:] - bar_width / 2, df['one_shot_f1_diff'], bar_width, label='One Shot - Zero Shot F1 Difference')

    # Plot the bars for ten_shot_f1_diff only if the value is not NA
    bars2 = ax.bar(indices[1:] + bar_width / 2, df['ten_shot_f1_diff'].where(df['ten_shot_f1_diff'].notna(), 0),
                   bar_width, label='Ten Shot - Zero Shot F1 Difference', color='orange')

    # Set the x-axis ticks and labels, including padding
    ax.set_xticks(indices)
    ax.set_xticklabels([''] + class_names, rotation=90)  # Add an empty label for the padding

    # Adding labels and title
    ax.set_xlabel('Class Unseen')
    ax.set_ylabel('F1 Score Difference')
    if graph_prefix is not None:
      ax.set_title(graph_prefix + '\n' + graph_name)
    else:
      ax.set_title(graph_name)


    # Adding custom x-axis at the center
    ax.spines['left'].set_position('zero')
    ax.spines['right'].set_color('none')
    ax.spines['bottom'].set_color('none')
    ax.spines['top'].set_color('none')
    ax.xaxis.tick_bottom()

    # Add the legend
    ax.legend()

    # Adding grid for better readability
    ax.yaxis.grid(True)

    # Display the plot
    plt.show()

# Only show ten shot available data

# Call the function with the dataframe
graph_name = 'F1 Score Differences for One Shot and Ten Shot With Relatives'
plot_grouped_bar_chart(df_shot_comp_grouped_with_relatives, graph_name)

graph_name = 'F1 Score Differences for One Shot and Ten Shot Without Relatives'
plot_grouped_bar_chart(df_shot_comp_grouped_without_relatives, graph_name)


In [ ]:
tag_relation_mapping = {
    'Ekosistem': {'type': 'top', 'hypernyms': [], 'hyponyms': ['Karasal Ekosistem', 'Yerleşim Yerleri', 'Sucul Ekosistem'], 'siblings': []},
    'Karasal Ekosistem': {'type': 'middle', 'hypernyms': ['Ekosistem'], 'hyponyms': ['Yerleşim Yerleri'], 'siblings': ['Sucul Ekosistem']},
    'Yerleşim Yerleri': {'type': 'bottom', 'hypernyms': ['Ekosistem', 'Karasal Ekosistem'], 'hyponyms': [], 'siblings': []},
    'Sucul Ekosistem': {'type': 'bottom', 'hypernyms': ['Ekosistem'], 'hyponyms': [], 'siblings': ['Karasal Ekosistem']},
    'Kirletici': {'type': 'top', 'hypernyms': [], 'hyponyms': ['Sıvı Kirletici', 'Katı Kirletici', 'Gaz Kirletici',  'Enerji'], 'siblings': []},
    'Sıvı Kirletici': {'type': 'bottom', 'hypernyms': ['Kirletici'], 'hyponyms': [], 'siblings': ['Katı Kirletici', 'Gaz Kirletici', 'Enerji']},
    'Katı Kirletici': {'type': 'bottom', 'hypernyms': ['Kirletici'], 'hyponyms': [], 'siblings': ['Sıvı Kirletici', 'Gaz Kirletici', 'Enerji']},
    'Gaz Kirletici': {'type': 'bottom', 'hypernyms': ['Kirletici'], 'hyponyms': [], 'siblings': ['Sıvı Kirletici', 'Katı Kirletici', 'Enerji']},
    'Enerji': {'type': 'bottom', 'hypernyms': ['Kirletici'], 'hyponyms': [], 'siblings': ['Sıvı Kirletici', 'Katı Kirletici', 'Gaz Kirletici']},
    'Afet': {'type': 'top', 'hypernyms': [], 'hyponyms': ['Doğal Afet', 'İnsan Kaynaklı Afet'], 'siblings': []},
    'Doğal Afet': {'type': 'bottom', 'hypernyms': ['Afet'], 'hyponyms': [], 'siblings': ['İnsan Kaynaklı Afet']},
    'İnsan Kaynaklı Afet': {'type': 'bottom', 'hypernyms': ['Afet'], 'hyponyms': [], 'siblings': ['Doğal Afet']},
    'Biota': {'type': 'top', 'hypernyms': [], 'hyponyms': ['İnsan Dışı Biota', 'Sucul Biota', 'Karasal Biota', 'İnsan'], 'siblings': []},
    'İnsan Dışı Biota': {'type': 'middle', 'hypernyms': ['Biota'], 'hyponyms': ['Sucul Biota', 'Karasal Biota'], 'siblings': ['İnsan']},
    'Sucul Biota': {'type': 'bottom', 'hypernyms': ['İnsan Dışı Biota', 'Biota'], 'hyponyms': [], 'siblings': ['Karasal Biota']},
    'Karasal Biota': {'type': 'bottom', 'hypernyms': ['İnsan Dışı Biota', 'Biota'], 'hyponyms': [], 'siblings': ['Sucul Biota']},
    'İnsan': {'type': 'bottom', 'hypernyms': ['Biota'], 'hyponyms': [], 'siblings': ['İnsan Dışı Biota']},
    'Çevresel Etki': {'type': 'top', 'hypernyms': [], 'hyponyms': ['Ekolojik Etki', 'Refah Etkisi', 'Ekonomik Etki', 'Sağlık Etkisi', 'Sosyal Etki'], 'siblings': []},
    'Ekolojik Etki': {'type': 'bottom', 'hypernyms': ['Çevresel Etki'], 'hyponyms': [], 'siblings': ['Refah Etkisi']},
    'Refah Etkisi': {'type': 'middle', 'hypernyms': ['Çevresel Etki'], 'hyponyms': ['Ekonomik Etki', 'Sağlık Etkisi', 'Sosyal Etki'], 'siblings': ['Ekolojik Etki']},
    'Ekonomik Etki': {'type': 'bottom', 'hypernyms': ['Refah Etkisi', 'Çevresel Etki'], 'hyponyms': [], 'siblings': ['Sağlık Etkisi',  'Sosyal Etki']},
    'Sağlık Etkisi': {'type': 'bottom', 'hypernyms': ['Refah Etkisi', 'Çevresel Etki'], 'hyponyms': [], 'siblings': ['Ekonomik Etki',  'Sosyal Etki']},
    'Sosyal Etki': {'type': 'bottom', 'hypernyms': ['Refah Etkisi', 'Çevresel Etki'], 'hyponyms': [], 'siblings': ['Ekonomik Etki',  'Sağlık Etkisi']},
    'Çevre Yönetimi': {'type': 'top', 'hypernyms': [], 'hyponyms': ['Düzenleme', 'Azaltma', 'Arıtım'], 'siblings': []},
    'Düzenleme': {'type': 'bottom', 'hypernyms': ['Çevre Yönetimi'], 'hyponyms': [], 'siblings': ['Azaltma', 'Arıtım']},
    'Azaltma': {'type': 'bottom', 'hypernyms': ['Çevre Yönetimi'], 'hyponyms': [], 'siblings': ['Düzenleme', 'Arıtım']},
    'Arıtım': {'type': 'bottom', 'hypernyms': ['Çevre Yönetimi'], 'hyponyms': [], 'siblings': ['Düzenleme', 'Azaltma']},
    'Kirleten': {'type': 'top', 'hypernyms': [], 'hyponyms': ['İnsan Kaynaklı Kirleten',  'Doğal Kirleten'], 'siblings': []},
    'İnsan Kaynaklı Kirleten': {'type': 'bottom', 'hypernyms': ['Kirleten'], 'hyponyms': [], 'siblings': ['Doğal Kirleten']},
    'Doğal Kirleten': {'type': 'bottom', 'hypernyms': ['Kirleten'], 'hyponyms': [], 'siblings': ['İnsan Kaynaklı Kirleten']}
}

# labels with no hypernyms
top_level_labels = []
# labels that have hypernyms and hyponyms
mid_level_labels = []
# labels with no hyponyms
bottom_level_labels = []
# labels with siblings
sibling_labels = []
# labels without siblings
no_sibling_labels = []

for tag in tag_relation_mapping:
  if len(tag_relation_mapping[tag]['hypernyms']) == 0:
    top_level_labels.append(entity_translation[tag])
  elif len(tag_relation_mapping[tag]['hyponyms']) == 0:
    bottom_level_labels.append(entity_translation[tag])
  else:
    mid_level_labels.append(entity_translation[tag])

  if len(tag_relation_mapping[tag]['siblings']) > 0:
    sibling_labels.append(entity_translation[tag])
  else:
    no_sibling_labels.append(entity_translation[tag])

print(top_level_labels)
print(mid_level_labels)
print(bottom_level_labels)
print(sibling_labels)
print(no_sibling_labels)

## Compare Top Level Labels Hyponym Results

Since top level only have hyponyms, these results show the performance of the model with and without hyponyms.

In [ ]:
def top_level_apply_compare(dataframe):
  removed_hyponyms = dataframe['removed_hyponyms']
  without_hyponym_results = dataframe[removed_hyponyms].iloc[0]
  with_hyponym_results = dataframe[~removed_hyponyms].iloc[0]

  return pd.Series({
      'test_length': dataframe.iloc[0]['test_length'],
      'without_hyponym_precision': without_hyponym_results['precision'],
      'with_hyponym_precision': with_hyponym_results['precision'],
      'hyponym_precision_diff': with_hyponym_results['precision'] - without_hyponym_results['precision'],
      'without_hyponym_accuracy': without_hyponym_results['accuracy'],
      'with_hyponym_accuracy': with_hyponym_results['accuracy'],
      'hyponym_accuracy_diff': with_hyponym_results['accuracy'] - without_hyponym_results['accuracy'],
      'without_hyponym_recall': without_hyponym_results['recall'],
      'with_hyponym_recall': with_hyponym_results['recall'],
      'hyponym_recall_diff': with_hyponym_results['recall'] - without_hyponym_results['recall'],
      'without_hyponym_f1': without_hyponym_results['f1'],
      'with_hyponym_f1': with_hyponym_results['f1'],
      'hyponym_f1_diff': with_hyponym_results['f1'] - without_hyponym_results['f1'],
  })


df_top = df[df['class_unseen'].isin(top_level_labels)]
df_top_result = df_top.groupby(['class_unseen', 'shot_number']).apply(top_level_apply_compare)
df_top_result
#df_top_result.to_excel("/content/drive/MyDrive/ner_results/top_results.xlsx")


In [ ]:
df_top_result = df_top_result.reset_index(level=['class_unseen','shot_number'])


In [ ]:
df_top_result_simple = df_top_result[['class_unseen','shot_number', 'without_hyponym_f1', 'with_hyponym_f1']]
df_top_result_simple

In [ ]:
df_top_result_simple.describe()

In [ ]:
print('HYPONYM T_TEST RESULTS')
print('ALL with hyponym vs without hyponym', t_test(df_top_result_simple, 'with_hyponym_f1', 'without_hyponym_f1'))
print('ZERO_SHOT with hyponym vs without hyponym', t_test(df_top_result_simple[df_top_result_simple['shot_number'] == 0], 'with_hyponym_f1', 'without_hyponym_f1'))
print('ONE_SHOT with hyponym vs without hyponym', t_test(df_top_result_simple[df_top_result_simple['shot_number'] == 1], 'with_hyponym_f1', 'without_hyponym_f1'))
print('TEN_SHOT with hyponym vs without hyponym', t_test(df_top_result_simple[df_top_result_simple['shot_number'] == 10], 'with_hyponym_f1', 'without_hyponym_f1'))

In [ ]:
pd.DataFrame([
    ['All Shots'] + list(t_test(df_top_result_simple, 'with_hyponym_f1', 'without_hyponym_f1')),
    ['Zero Shot'] + list(t_test(df_top_result_simple[df_top_result_simple['shot_number'] == 0], 'with_hyponym_f1', 'without_hyponym_f1')),
    ['One Shot'] + list(t_test(df_top_result_simple[df_top_result_simple['shot_number'] == 1], 'with_hyponym_f1', 'without_hyponym_f1')),
    ['Ten Shot'] + list(t_test(df_top_result_simple[df_top_result_simple['shot_number'] == 10], 'with_hyponym_f1', 'without_hyponym_f1')),
    ], columns=['Setup', 'T Test', 'P Value'])

In [ ]:
def plot_vertical_bar_chart(data, colors=None, title="Vertical Bar Chart", ylabel="Values", padding_percent=10, bar_width_factor=0.8, bar_labels=None):
    """
    Plots a vertical bar chart with:
      - Each row representing a category
      - A centered, bolder zero line
      - Transparent dividers between categories
      - Supports any number of bars per category, with consistent width and no overlap
      - One legend entry per color
      - Vertical x-axis title
      - Centered title
      - X-axis labels centered to the middle bar of their data
      - Two-word x-axis labels split into two lines
      - Legend placed inside the graph
      - Second and third bars swapped
      - Graph scaled to prevent legend overlap

    Args:
        data (dict): A dictionary where keys are category names and values are lists of numeric values.
        colors (list, optional): A list of colors for the bars. Defaults to a predefined color cycle.
        title (str, optional): The title of the chart.
        ylabel (str, optional): The label for the y-axis.
        padding_percent (float, optional): Percentage of padding around min/max values. Default is 10%.
        bar_width_factor (float, optional): Factor to adjust the width of the bars (0.0 to 1.0). Default is 0.8.
        bar_labels (list, optional): A list of labels for each bar color. Defaults to ["Value 1", "Value 2", ...].
    """

    categories = list(data.keys())
    values = list(data.values())

    # Swap second and third bars
    values[1], values[2] = values[2], values[1]
    categories[1], categories[2] = categories[2], categories[1]

    num_categories = len(categories)
    max_num_columns = max(len(val) for val in values)

    # Colors (if not provided, use a colormap for dynamic color generation)
    if colors is None:
        cmap = plt.cm.get_cmap('Set1')
        colors = cmap(np.linspace(0, 1, max_num_columns))

    # Bar labels (if not provided, use default labels)
    if bar_labels is None:
        bar_labels = [f"Value {i+1}" for i in range(max_num_columns)]

    # Create Plot
    fig, ax = plt.subplots(figsize=(10, 6))

    # Bar Positions and Width
    bar_width = 0.8 / max_num_columns * bar_width_factor
    index = np.arange(num_categories)

    # Calculate centered x-tick positions
    centered_xticks = []
    for i, cat_values in enumerate(values):
        num_bars = len(cat_values)
        center_offset = (num_bars - 1) * bar_width / 2
        centered_xticks.append(index[i] + center_offset)

    # Determine Limits with Padding
    min_val = min([min(val) for val in values if val])
    max_val = max([max(val) for val in values if val])
    max_abs_val = max(abs(min_val), abs(max_val))
    padding = max_abs_val * (padding_percent / 100)
    ylim = (0, max_abs_val + padding)  # Start from 0

    # Plot Bars and Create Legend Entries
    legend_elements = []
    for i in range(max_num_columns):
        bar = ax.bar(0, 0, width=bar_width, color=colors[i], label=bar_labels[i])
        legend_elements.append(bar)
        for j, cat_values in enumerate(values):
            if i < len(cat_values):
                ax.bar(index[j] + i * bar_width, cat_values[i], width=bar_width, color=colors[i])

    # Centered Zero Line
    ax.axhline(0, color='black', linestyle='-', linewidth=2)

    # Category Dividers
    divider_positions = index + 0.8
    for pos in divider_positions[:-1]:
        ax.axvline(pos, color='gray', linestyle=':', alpha=0.5)

    # Adjust plot area to prevent legend overlap
    box = ax.get_position()
    ax.set_position([box.x0, box.y0, box.width * 0.85, box.height])  # Increased width slightly

    # Format x-axis labels to split two-word labels into two lines
    formatted_categories = []
    for cat in categories:
        words = cat.split()
        if len(words) == 2:
            formatted_categories.append(f"{words[0]}\n{words[1]}")
        else:
            formatted_categories.append(cat)

    # Customize Plot
    ax.set_ylabel(ylabel)
    if graph_prefix is not None:
      full_title = graph_prefix + '\n' + title
    else:
      full_title = title
    ax.set_title(full_title, ha='center')

    ax.set_xticks(centered_xticks)
    ax.set_xticklabels(formatted_categories, rotation='vertical')
    ax.set_ylim(ylim)

    # Place legend inside the graph
    ax.legend(loc='upper right', title='Values')

    ax.grid(axis='y', linestyle='--')

    # Show Plot
    plt.show()

In [ ]:
def plot_df(dataframe, column_names, data_arr):
  shot_number = dataframe.iloc[0]['shot_number']
  class_names = dataframe['class_unseen']

  data = {}
  for cls in class_names:
    s = dataframe[dataframe['class_unseen'] == cls].iloc[0]
    data[cls] = s[column_names].array

  data_arr.append({
      'shot_number': shot_number,
      'data': data
      })

result_types = [
    'precision',
    'accuracy',
    'recall',
    'f1'
]

In [ ]:
parameter = 'hyponym'

for result_type in result_types:
  arr = []

  df_top_result.groupby('shot_number').apply(plot_df, column_names=[f'without_{parameter}_{result_type}', f'{parameter}_{result_type}_diff', f'with_{parameter}_{result_type}', ], data_arr=arr)

  for d in arr:
    shot_name = 'Zero Shot'
    if d['shot_number'] == 1:
      shot_name = 'One Shot'
    elif d['shot_number'] == 10:
      shot_name = 'Ten Shots'

    plot_vertical_bar_chart(d['data'], colors=['#5B99C2', '#FF8343', '#387F39'], title=f"{parameter.capitalize()} {shot_name} {result_type.capitalize()} Results", ylabel=result_type.capitalize(), padding_percent=10, bar_width_factor=0.8,
                              bar_labels=[f'Without {parameter.capitalize()} {result_type.capitalize()}', f'{parameter.capitalize()} {result_type.capitalize()} Difference', f'With {parameter.capitalize()} {result_type.capitalize()}', ])

## Compare Bottom Level Labels Hypernym Results
Since bottom level label can have siblings, in this comparison siblings situtation are not taken into account. It means, siblings are not removed from the dataset. To see affect of siblings, check other results.

In [ ]:
def bottom_level_apply_compare(dataframe):
  class_unseen = dataframe.iloc[0]['class_unseen']
  has_sibling = class_unseen in sibling_labels

  removed_hypernyms = dataframe['removed_hypernyms']
  removed_siblings = dataframe['removed_siblings']
  ignore_siblings = ((has_sibling & ~removed_siblings) | (~has_sibling & removed_siblings))

  without_hypernym_results = dataframe[removed_hypernyms & ignore_siblings].iloc[0]
  with_hypernym_results = dataframe[~removed_hypernyms & ignore_siblings].iloc[0]

  return pd.Series({
      'test_length': dataframe.iloc[0]['test_length'],
      'without_hypernym_precision': without_hypernym_results['precision'],
      'with_hypernym_precision': with_hypernym_results['precision'],
      'hypernym_precision_diff': with_hypernym_results['precision'] - without_hypernym_results['precision'],
      'without_hypernym_accuracy': without_hypernym_results['accuracy'],
      'with_hypernym_accuracy': with_hypernym_results['accuracy'],
      'hypernym_accuracy_diff': with_hypernym_results['accuracy'] - without_hypernym_results['accuracy'],
      'without_hypernym_recall': without_hypernym_results['recall'],
      'with_hypernym_recall': with_hypernym_results['recall'],
      'hypernym_recall_diff': with_hypernym_results['recall'] - without_hypernym_results['recall'],
      'without_hypernym_f1': without_hypernym_results['f1'],
      'with_hypernym_f1': with_hypernym_results['f1'],
      'hypernym_f1_diff': with_hypernym_results['f1'] - without_hypernym_results['f1']
  })


df_bottom = df[df['class_unseen'].isin(bottom_level_labels)]
df_bottom = df_bottom.groupby(['class_unseen', 'shot_number']).apply(bottom_level_apply_compare)
#df_bottom.to_excel("/content/drive/MyDrive/ner_results/bottom_results.xlsx")
df_bottom




In [ ]:
df_bottom_result = df_bottom.reset_index(level=['class_unseen','shot_number'])

In [ ]:
df_bottom_result_simple = df_bottom_result[['class_unseen','shot_number', 'without_hypernym_f1', 'with_hypernym_f1']]
df_bottom_result_simple

In [ ]:
df_bottom_result_simple.describe()

In [ ]:
print('HYPERNYM T_TEST RESULTS')
print('ALL with hypernym vs without hypernym', t_test(df_bottom_result_simple, 'with_hypernym_f1', 'without_hypernym_f1'))
print('ZERO_SHOT with hypernym vs without hypernym', t_test(df_bottom_result_simple[df_bottom_result_simple['shot_number'] == 0], 'with_hypernym_f1', 'without_hypernym_f1'))
print('ONE_SHOT with hypernym vs without hypernym', t_test(df_bottom_result_simple[df_bottom_result_simple['shot_number'] == 1], 'with_hypernym_f1', 'without_hypernym_f1'))
print('TEN_SHOT with hypernym vs without hypernym', t_test(df_bottom_result_simple[df_bottom_result_simple['shot_number'] == 10], 'with_hypernym_f1', 'without_hypernym_f1'))

In [ ]:
pd.DataFrame([
    ['All Shots'] + list(t_test(df_bottom_result_simple, 'with_hypernym_f1', 'without_hypernym_f1')),
    ['Zero Shot'] + list(t_test(df_bottom_result_simple[df_bottom_result_simple['shot_number'] == 0], 'with_hypernym_f1', 'without_hypernym_f1')),
    ['One Shot'] + list(t_test(df_bottom_result_simple[df_bottom_result_simple['shot_number'] == 1], 'with_hypernym_f1', 'without_hypernym_f1')),
    ['Ten Shot'] + list(t_test(df_bottom_result_simple[df_bottom_result_simple['shot_number'] == 10], 'with_hypernym_f1', 'without_hypernym_f1')),
    ], columns=['Setup', 'T Test', 'P Value'])

In [ ]:
parameter = 'hypernym'
#df_bottom_result = df_bottom


for result_type in result_types:
  arr = []

  df_bottom_result.groupby('shot_number').apply(plot_df, column_names=[f'without_{parameter}_{result_type}', f'{parameter}_{result_type}_diff', f'with_{parameter}_{result_type}', ], data_arr=arr)

  for d in arr:
    shot_name = 'Zero Shot'
    if d['shot_number'] == 1:
      shot_name = 'One Shot'
    elif d['shot_number'] == 10:
      shot_name = 'Ten Shots'

    plot_vertical_bar_chart(d['data'], colors=['#5B99C2', '#FF8343', '#387F39'], title=f"{parameter.capitalize()} {shot_name} {result_type.capitalize()} Results", ylabel=result_type.capitalize(), padding_percent=10, bar_width_factor=0.8,
                              bar_labels=[f'Without {parameter.capitalize()} {result_type.capitalize()}', f'{parameter.capitalize()} {result_type.capitalize()} Difference', f'With {parameter.capitalize()} {result_type.capitalize()}'])

In [ ]:
def mid_level_apply_compare(dataframe):
  class_unseen = dataframe.iloc[0]['class_unseen']
  has_sibling = class_unseen in sibling_labels

  removed_hypernyms = dataframe['removed_hypernyms']
  removed_hyponyms = dataframe['removed_hyponyms']
  removed_siblings = dataframe['removed_siblings']
  ignore_siblings = ((has_sibling & ~removed_siblings) | (~has_sibling & removed_siblings))

  without_both_results = dataframe[removed_hypernyms & removed_hyponyms & ignore_siblings].iloc[0]
  without_hypernym_results = dataframe[removed_hypernyms & ~removed_hyponyms & ignore_siblings].iloc[0]
  without_hyponym_results = dataframe[~removed_hypernyms & removed_hyponyms & ignore_siblings].iloc[0]
  with_all = dataframe[~removed_hypernyms & ~removed_hyponyms & ignore_siblings].iloc[0]

  return pd.Series({
      'test_length': dataframe.iloc[0]['test_length'],
      'without_both_precision': without_both_results['precision'],
      'with_hyponym_precision': without_hypernym_results['precision'],
      'with_hypernym_precision': without_hyponym_results['precision'],
      'with_both_precision': with_all['precision'],
      'hyponym_precision_difference': without_hypernym_results['precision'] - without_both_results['precision'],
      'hypernym_precision_difference': without_hyponym_results['precision'] - without_both_results['precision'],
      'both_precision_difference': with_all['precision'] - without_both_results['precision'],
      'without_both_accuracy': without_both_results['accuracy'],
      'with_hyponym_accuracy': without_hypernym_results['accuracy'],
      'with_hypernym_accuracy': without_hyponym_results['accuracy'],
      'with_both_accuracy': with_all['accuracy'],
      'hyponym_accuracy_difference': without_hypernym_results['accuracy'] - without_both_results['accuracy'],
      'hypernym_accuracy_difference': without_hyponym_results['accuracy'] - without_both_results['accuracy'],
      'both_accuracy_difference': with_all['accuracy'] - without_both_results['accuracy'],
      'without_both_recall': without_both_results['recall'],
      'with_hyponym_recall': without_hypernym_results['recall'],
      'with_hypernym_recall': without_hyponym_results['recall'],
      'with_both_recall': with_all['recall'],
      'hyponym_recall_difference': without_hypernym_results['recall'] - without_both_results['recall'],
      'hypernym_recall_difference': without_hyponym_results['recall'] - without_both_results['recall'],
      'both_recall_difference': with_all['recall'] - without_both_results['recall'],
      'without_both_f1': without_both_results['f1'],
      'with_hyponym_f1': without_hypernym_results['f1'],
      'with_hypernym_f1': without_hyponym_results['f1'],
      'with_both_f1': with_all['f1'],
      'hyponym_f1_difference': without_hypernym_results['f1'] - without_both_results['f1'],
      'hypernym_f1_difference': without_hyponym_results['f1'] - without_both_results['f1'],
      'both_f1_difference': with_all['f1'] - without_both_results['f1'],
  })




df_mid_level = df[df['class_unseen'].isin(mid_level_labels)].groupby(['class_unseen', 'shot_number']).apply(mid_level_apply_compare)
#df_mid_level.to_excel("/content/drive/MyDrive/ner_results/middle_results.xlsx")
df_mid_level

In [ ]:
df_mid_level_result = df_mid_level.reset_index(level=['class_unseen','shot_number'])

for result_type in result_types:
  column_names = [
    f'without_both_{result_type}',
    f'with_hyponym_{result_type}',
    f'hyponym_{result_type}_difference',
    f'without_both_{result_type}',
    f'with_hypernym_{result_type}',
    f'hypernym_{result_type}_difference',
    f'without_both_{result_type}',
    f'with_both_{result_type}',
    f'both_{result_type}_difference'
  ]

  bar_labels= [
    f'{result_type} Without Hyponym and Hypernym',
    f'{result_type} Only With Hyponym',
    f'Difference of {result_type} Only With Hyponym',
    f'{result_type} Without Hyponym and Hypernym',
    f'{result_type} Only With Hypernym',
    f'Difference of {result_type} Only With Hypernym',
    f'{result_type} Without Hyponym and Hypernym',
    f'{result_type} With Hyponym and Hypernym',
    f'Difference of {result_type} With Hyponym and Hypernym',
  ]

  arr = []

  df_mid_level_result.groupby('shot_number').apply(plot_df, column_names=column_names, data_arr=arr)

  for d in arr:
    shot_name = 'Zero Shot'
    if d['shot_number'] == 1:
      shot_name = 'One Shot'
    elif d['shot_number'] == 10:
      shot_name = 'Ten Shots'

    plot_horizontal_bar_chart(d['data'], colors=None,
                              title=f"Hyponym And Hypernym {shot_name} {result_type.capitalize()} Results",
                              xlabel=result_type.capitalize(), padding_percent=10, bar_width_factor=0.8,
                              bar_labels=bar_labels)

In [ ]:
df_mid_level_result_simple = df_mid_level_result[['class_unseen','shot_number', 'without_both_f1', 'with_both_f1']]
df_mid_level_result_simple

In [ ]:
df_mid_level_result_simple.describe()

In [ ]:
print('HYPONYM AND HYPERNYM T_TEST RESULTS')
print('ALL with hyponym and hypernym vs without hyponym and hypernym', t_test(df_mid_level_result_simple, 'with_both_f1', 'without_both_f1'))
print('ZERO_SHOT with hyponym and hypernym vs without hyponym and hypernym', t_test(df_mid_level_result_simple[df_mid_level_result_simple['shot_number'] == 0], 'with_both_f1', 'without_both_f1'))
print('ONE_SHOT with hyponym and hypernym vs without hyponym and hypernym', t_test(df_mid_level_result_simple[df_mid_level_result_simple['shot_number'] == 1], 'with_both_f1', 'without_both_f1'))
print('TEN_SHOT with hyponym and hypernym vs without hyponym and hypernym', t_test(df_mid_level_result_simple[df_mid_level_result_simple['shot_number'] == 10], 'with_both_f1', 'without_both_f1'))

In [ ]:
pd.DataFrame([
    ['All Shots'] + list(t_test(df_mid_level_result_simple, 'with_both_f1', 'without_both_f1')),
    ['Zero Shot'] + list(t_test(df_mid_level_result_simple[df_mid_level_result_simple['shot_number'] == 0], 'with_both_f1', 'without_both_f1')),
    ['One Shot'] + list(t_test(df_mid_level_result_simple[df_mid_level_result_simple['shot_number'] == 1], 'with_both_f1', 'without_both_f1')),
    ['Ten Shot'] + list(t_test(df_mid_level_result_simple[df_mid_level_result_simple['shot_number'] == 10], 'with_both_f1', 'without_both_f1')),
    ], columns=['Setup', 'T Test', 'P Value'])

In [ ]:
def sibling_compare_apply(dataframe):
  class_unseen = dataframe['class_unseen'].iloc[0]

  has_hypernym = class_unseen not in top_level_labels
  has_hyponym = class_unseen not in bottom_level_labels

  removed_hypernyms = dataframe['removed_hypernyms']
  removed_hyponyms = dataframe['removed_hyponyms']
  removed_siblings = dataframe['removed_siblings']

  ignore_hypernyms = (has_hypernym & ~removed_hypernyms) | (~has_hypernym & removed_hypernyms)
  ignore_hyponyms = (has_hyponym & ~removed_hyponyms) | (~has_hyponym & removed_hyponyms)

  without_sibling_results = dataframe[removed_siblings & ignore_hypernyms & ignore_hyponyms].iloc[0]
  with_sibling_results = dataframe[~removed_siblings & ignore_hypernyms & ignore_hyponyms].iloc[0]

  return pd.Series({
      'test_length': dataframe.iloc[0]['test_length'],
      'without_sibling_precision': without_sibling_results['precision'],
      'with_sibling_precision': with_sibling_results['precision'],
      'sibling_precision_difference': with_sibling_results['precision'] - without_sibling_results['precision'],
      'without_sibling_accuracy': without_sibling_results['accuracy'],
      'with_sibling_accuracy': with_sibling_results['accuracy'],
      'sibling_accuracy_difference': with_sibling_results['accuracy'] - without_sibling_results['accuracy'],
      'without_sibling_recall': without_sibling_results['recall'],
      'with_sibling_recall': with_sibling_results['recall'],
      'sibling_recall_difference': with_sibling_results['recall'] - without_sibling_results['recall'],
      'without_sibling_f1': without_sibling_results['f1'],
      'with_sibling_f1': with_sibling_results['f1'],
      'sibling_f1_difference': with_sibling_results['f1'] - without_sibling_results['f1']
  })




df_sibling = df[df['class_unseen'].isin(sibling_labels)].groupby(['class_unseen', 'shot_number']).apply(sibling_compare_apply)
#df_sibling.to_excel("/content/drive/MyDrive/ner_results/sibling_results.xlsx")
df_sibling

In [ ]:
parameter = 'sibling'
df_sibling_result = df_sibling.reset_index(level=['class_unseen','shot_number'])
df_sibling_result_zeroes = df_sibling_result[df_sibling_result['without_sibling_f1'] == 0]['class_unseen']
df_sibling_result = df_sibling_result[~df_sibling_result['class_unseen'].isin(df_sibling_result_zeroes)]
df_sibling_result

In [ ]:
df_sibling_result_simple = df_sibling_result[['class_unseen','shot_number', 'without_sibling_f1', 'with_sibling_f1']]
df_sibling_result_simple

In [ ]:
df_sibling_result_simple.describe()

In [ ]:
print("SIBLING T_TEST RESULTS")
print('ALL with sibling vs without sibling', t_test(df_sibling_result_simple, 'with_sibling_f1', 'without_sibling_f1'))
print('ZERO_SHOT with sibling vs without sibling', t_test(df_sibling_result_simple[df_sibling_result_simple['shot_number'] == 0], 'with_sibling_f1', 'without_sibling_f1'))
print('ONE_SHOT with sibling vs without sibling', t_test(df_sibling_result_simple[df_sibling_result_simple['shot_number'] == 1], 'with_sibling_f1', 'without_sibling_f1'))
print('TEN_SHOT with sibling vs without sibling', t_test(df_sibling_result_simple[df_sibling_result_simple['shot_number'] == 10], 'with_sibling_f1', 'without_sibling_f1'))
print('TEN_SHOT without sibling vs with sibling', t_test(df_sibling_result_simple[df_sibling_result_simple['shot_number'] == 10], 'without_sibling_f1', 'with_sibling_f1'))
print('TEN_SHOT without sibling vs with sibling', t_test_two_way(df_sibling_result_simple[df_sibling_result_simple['shot_number'] == 10], 'without_sibling_f1', 'with_sibling_f1'))

In [ ]:
pd.DataFrame([
    ['All Shots'] + list(t_test(df_sibling_result_simple, 'with_sibling_f1', 'without_sibling_f1')),
    ['Zero Shot'] + list(t_test(df_sibling_result_simple[df_sibling_result_simple['shot_number'] == 0], 'with_sibling_f1', 'without_sibling_f1')),
    ['One Shot'] + list(t_test(df_sibling_result_simple[df_sibling_result_simple['shot_number'] == 1], 'with_sibling_f1', 'without_sibling_f1')),
    ['Ten Shot'] + list(t_test(df_sibling_result_simple[df_sibling_result_simple['shot_number'] == 10], 'with_sibling_f1', 'without_sibling_f1')),
    ['Ten Shot Reversed'] + list(t_test(df_sibling_result_simple[df_sibling_result_simple['shot_number'] == 10], 'without_sibling_f1', 'with_sibling_f1')),
    ], columns=['Setup', 'T Test', 'P Value'])

In [ ]:
for result_type in result_types:
  arr = []

  df_sibling_result.groupby('shot_number').apply(plot_df, column_names=[f'without_{parameter}_{result_type}', f'with_{parameter}_{result_type}', f'{parameter}_{result_type}_difference'], data_arr=arr)

  for d in arr:
    shot_name = 'Zero Shot'
    if d['shot_number'] == 1:
      shot_name = 'One Shot'
    elif d['shot_number'] == 10:
      shot_name = 'Ten Shots'

    plot_horizontal_bar_chart(d['data'], colors=None, title=f"{parameter.capitalize()} {shot_name} {result_type.capitalize()} Results", xlabel=result_type.capitalize(), padding_percent=10, bar_width_factor=0.8,
                              bar_labels=[f'Without {parameter.capitalize()} {result_type.capitalize()}', f'With {parameter.capitalize()} {result_type.capitalize()}', f'{parameter.capitalize()} {result_type.capitalize()} Difference'])